In [1]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

import numpy as np
import pandas as pd
from common.const import PRODUCT_VECTOR_DB_COLLECTION, DEVICE, SCHEMA, TABLE_RAW_PRODUCTS, TABLE_RAW_REVIEWS
from common.helper import clickhouse_extract_data_as_df
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue, Range
import faiss
from common.const import SCHEMA, TABLE_RAW_PRODUCTS, DEVICE
from common.helper import clickhouse_extract_data_as_df


c:\Users\Admin\.conda\envs\scc454\lib\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: '[WinError 127] The specified procedure could not be found'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [2]:
# Query data

def query_products_data(n_rows, offset):
    products_query = f"""
    SELECT
        parent_asin,
        title,
        description,
        features,
        main_category,
        details,
        CAST(price AS Nullable(Float64)) AS price,
        average_rating,
        rating_number
    FROM {SCHEMA}.{TABLE_RAW_PRODUCTS}
    LIMIT {n_rows} OFFSET {offset}
    """
    return clickhouse_extract_data_as_df(products_query)

def query_reviews_data(n_rows, offset):
    reviews_query = f"""
    SELECT 
        user_id,
        parent_asin,
        title,
        text,
        rating
    FROM {SCHEMA}.{TABLE_RAW_REVIEWS}
    LIMIT {n_rows} OFFSET {offset}
    """
    return clickhouse_extract_data_as_df(reviews_query)

In [3]:
def create_product_text(row):
    text_parts = []

    if pd.notna(row['title']):
        text_parts.append((row['title'] + " ") * 1 + row['title']) # duplicate once

    if pd.notna(row['description']):
        text_parts.append(row['description'])

    if pd.notna(row['features']):
        text_parts.append(row['features'])

    if pd.notna(row['main_category']):
        text_parts.append((row['main_category'] + " ") * 2 + row['main_category'])
    
    if pd.notna(row['details']):
        text_parts.append(row['details'])

    return ' '.join(text_parts)

In [4]:
products_df = query_products_data(50000, 0)

In [5]:
products_df[products_df["parent_asin"]=="B07484MS4G"]

,parent_asin,title,description,features,main_category,details,price,average_rating,rating_number


In [6]:
#test multiple query products (query 1)
from itertools import product
import time

from sympy import product


def precision_at_k(I_exact, I_approx):
    matches = [
        len(set(e).intersection(set(a)))
        for e, a in zip(I_exact, I_approx)
    ]
    return np.mean(np.array(matches) / I_exact.shape[1])

def mean_pm_2std(values):
    values = np.array(values)
    mean = np.mean(values)
    std = np.std(values, ddof=1)  # sample std
    lower = mean - 2 * std
    upper = mean + 2 * std
    return mean, std, lower, upper



model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
product_query_offset = 0
review_query_offset = 0
BATCH_SIZE = 500
for dataset_size in [50000,500000]:
    for vector_dim in [384]:
        for m in [8, 16, 32]:
            if m == 8:
                ef_construction = 50
                ef_search = 32
            elif m == 16:
                ef_construction = 100
                ef_search = 64
            elif m == 32:
                ef_construction = 200
                ef_search = 128
            ### START PRODUCT SIM

            print(f"Precision@10 for dataset size {dataset_size}, vector dim {vector_dim}, m {m}, ef_construction {ef_construction}, ef_search {ef_search}")
            products_df = query_products_data(dataset_size, product_query_offset)
            products_df['combined_text'] = products_df.apply(create_product_text, axis=1)
            product_embeddings = model.encode(
                products_df['combined_text'].tolist(),
                show_progress_bar=True,
                batch_size=32,
                normalize_embeddings=True,  # Normalize for cosine similarity
                device=DEVICE
            )

            # Create index once
            index_exact = faiss.IndexIDMap(faiss.IndexFlatIP(vector_dim))
            for i in range(0, len(product_embeddings), BATCH_SIZE): #batch insert exact index
                batch_vectors = product_embeddings[i:i+BATCH_SIZE].astype("float32")

                # Use dataframe index as semantic ID
                batch_ids = products_df.index[i:i+len(batch_vectors)].to_numpy().astype("int64")

                index_exact.add_with_ids(batch_vectors, batch_ids)

                # Save after each batch
            filename = f"product_exact_{dataset_size}_{vector_dim}_m{m}_efc{ef_construction}_efs{ef_search}.index"
            faiss.write_index(index_exact, filename)
            
            base_hnsw  = faiss.IndexHNSWSQ(vector_dim, faiss.ScalarQuantizer.QT_8bit, m, faiss.METRIC_INNER_PRODUCT)

            # Set ef_construction and ef_search parameters
            base_hnsw.hnsw.efConstruction = ef_construction
            base_hnsw.hnsw.efSearch = ef_search
            base_hnsw.train(product_embeddings.astype("float32"))
            index_hnsw = faiss.IndexIDMap(base_hnsw)

            for i in range(0, len(product_embeddings), BATCH_SIZE): #batch insert hnsw index
                batch_vectors = product_embeddings[i:i+BATCH_SIZE].astype("float32")

                # Use dataframe index as semantic ID
                batch_ids = products_df.index[i:i+len(batch_vectors)].to_numpy().astype("int64")

                index_hnsw.add_with_ids(batch_vectors, batch_ids)

                # Save after each batch
            filename = f"product_hnsw_{dataset_size}_{vector_dim}_m{m}_efc{ef_construction}_efs{ef_search}.index"
            faiss.write_index(index_hnsw, filename)

            k = 11
            exact_times = []
            approximate_times = []
            p10s = []
            product_asins = ['B07484MS4G', 'B01FYDF3VI', 'B01C3PMM7S', 'B012HUTKTM', 'B075R6NQ1W', 'B08DCZGQJ5', 'B077WBYPLG', 'B001KYU8UM', 'B0919HCJDN', 'B08MQCXGJM']
            test_product_query = f"""
                    SELECT
                        parent_asin,
                        title,
                        description,
                        features,
                        main_category,
                        details,
                        CAST(price AS Nullable(Float64)) AS price,
                        average_rating,
                        rating_number
                    FROM {SCHEMA}.{TABLE_RAW_PRODUCTS}
                    WHERE parent_asin IN {tuple(product_asins)}
                    """
            test_products_df = clickhouse_extract_data_as_df(test_product_query)
            test_products_df['combined_text'] = test_products_df.apply(create_product_text, axis=1)
            test_product_embeddings = model.encode(
                test_products_df['combined_text'].tolist(),
                show_progress_bar=True,
                batch_size=32,
                normalize_embeddings=True,  # Normalize for cosine similarity
                device=DEVICE
            )
            for test_product_embedding in test_product_embeddings:
                query_vector = test_product_embedding.reshape(1, -1).astype("float32")
                start_time = time.time()
                D_exact, I_exact = index_exact.search(query_vector, k)
                exact_times.append(time.time() - start_time)

                start_time = time.time()
                D_hnsw, I_hnsw = index_hnsw.search(query_vector, k)
                approximate_times.append(time.time() - start_time)
                # Remove self-match (first column)
                I_exact = I_exact[:, 1:]
                I_hnsw = I_hnsw[:, 1:]

                p10s.append(precision_at_k(I_exact, I_hnsw))

            exact_mean, exact_std, exact_low, exact_high = mean_pm_2std(exact_times)
            approx_mean, approx_std, approx_low, approx_high = mean_pm_2std(approximate_times)
            p10_mean, p10_std, p10_low, p10_high = mean_pm_2std(p10s)

            print(f"PRODUCT {dataset_size} {m} {ef_construction} {ef_search} Precision@10: {p10_mean:.4f} ± {2*p10_std:.4f} "
                f"(range: {p10_low:.4f} - {p10_high:.4f})")
            
            print(f"PRODUCT {dataset_size} {m} {ef_construction} {ef_search} Exact Time: {exact_mean:.4f} ± {2*exact_std:.4f} "
                f"(range: {exact_low:.4f} - {exact_high:.4f})")

            print(f"PRODUCT {dataset_size} {m} {ef_construction} {ef_search} Approx Time: {approx_mean:.4f} ± {2*approx_std:.4f} "
                f"(range: {approx_low:.4f} - {approx_high:.4f})")

            # END PRODUCT SIM

            # START USER SIM
            # Save user reviews for later use
            reviews_df = query_reviews_data(dataset_size, review_query_offset)
            reviews_df['full_review'] = (
                reviews_df['title'].fillna('') + ' ' +
                reviews_df['text'].fillna('')
            )

            user_reviews = (
                reviews_df
                .groupby('user_id')['full_review']
                .apply(' '.join)
                .reset_index(name='all_reviews_text')
            )

            user_embeddings = model.encode(
                user_reviews['all_reviews_text'].tolist(),
                show_progress_bar=True,
                batch_size=32,
                normalize_embeddings=True,  # Normalize for cosine similarity
                device=DEVICE
            )
            id_to_index = dict(zip(user_reviews["user_id"], user_reviews.index)) #to get fast index from asin for querying (o1 lookup)
            index_exact = faiss.IndexIDMap(faiss.IndexFlatIP(vector_dim))
            for i in range(0, len(user_embeddings), BATCH_SIZE): #batch insert exact index
                batch_vectors = user_embeddings[i:i+BATCH_SIZE].astype("float32")

                # Use dataframe index as semantic ID
                batch_ids = user_reviews.index[i:i+len(batch_vectors)].to_numpy().astype("int64")

                index_exact.add_with_ids(batch_vectors, batch_ids)

                # Save after each batch
            filename = f"user_exact_{dataset_size}_{vector_dim}_m{m}_efc{ef_construction}_efs{ef_search}.index"
            faiss.write_index(index_exact, filename)
            
            base_hnsw = faiss.IndexHNSWSQ(vector_dim, faiss.ScalarQuantizer.QT_8bit, m, faiss.METRIC_INNER_PRODUCT)

            # Set ef_construction and ef_search parameters
            base_hnsw.hnsw.efConstruction = ef_construction
            base_hnsw.hnsw.efSearch = ef_search

            base_hnsw.train(product_embeddings.astype("float32"))
            index_hnsw = faiss.IndexIDMap(base_hnsw)

            for i in range(0, len(user_embeddings), BATCH_SIZE): #batch insert hnsw index
                batch_vectors = user_embeddings[i:i+BATCH_SIZE].astype("float32")

                # Use dataframe index as semantic ID
                batch_ids = user_reviews.index[i:i+len(batch_vectors)].to_numpy().astype("int64")

                index_hnsw.add_with_ids(batch_vectors, batch_ids)

                # Save after each batch
            filename = f"user_hnsw_{dataset_size}_{vector_dim}_m{m}_efc{ef_construction}_efs{ef_search}.index"
            faiss.write_index(index_hnsw, filename)

            k = 11
            exact_times = []
            approximate_times = []
            p10s = []
            random_users = ['AG5JJZKUDTC5PREHPTI5NOC75M2Q', 'AFQIHDRZZ7RGRLSFKDGCYPBHHTKA', 'AFXCPSCWLOSVQZHN7FAECOYAT3RA', 'AHT6VS3G3JSA7KBOCBT6KPUCJX7A', 
                           'AH4G4SNKSIWZNGZ5EDNJTRORKIMQ', 'AG4JLUSDCYTYSSE5SQADXOPATLTA', 'AGXVAAK6XJ7C2GUTYQEMBOGQ7JGQ', 'AGKPEXI6SAACYVXRNMICW3NKG2XA', 
                           'AHA272AOVKWZUXL672G653A2MTIA', 'AH6X5VUV5AUGO3G55G7AL6GZ2QZQ']
            test_user_query = f"""
                SELECT 
                    user_id,
                    parent_asin,
                    title,
                    text,
                    rating
                FROM {SCHEMA}.{TABLE_RAW_REVIEWS}
                WHERE user_id IN {tuple(random_users)}
                """
            test_reviews_df = clickhouse_extract_data_as_df(test_user_query)
            test_reviews_df['full_review'] = (
                test_reviews_df['title'].fillna('') + ' ' +
                test_reviews_df['text'].fillna('')
            )

            test_user_reviews = (
                test_reviews_df
                .groupby('user_id')['full_review']
                .apply(' '.join)
                .reset_index(name='all_reviews_text')
            )
            test_user_embeddings = model.encode(
                test_user_reviews['all_reviews_text'].tolist(),
                show_progress_bar=True,
                batch_size=32,
                normalize_embeddings=True,  # Normalize for cosine similarity
                device=DEVICE
            )
            for test_user_embedding in test_user_embeddings:
                query_vector = test_user_embedding.reshape(1, -1).astype("float32")
                start_time = time.time()
                D_exact, I_exact = index_exact.search(query_vector, k)
                exact_times.append(time.time() - start_time)

                start_time = time.time()
                D_hnsw, I_hnsw = index_hnsw.search(query_vector, k)
                approximate_times.append(time.time() - start_time)
                # Remove self-match (first column)
                I_exact = I_exact[:, 1:]
                I_hnsw = I_hnsw[:, 1:]

                p10s.append(precision_at_k(I_exact, I_hnsw))

            exact_mean, exact_std, exact_low, exact_high = mean_pm_2std(exact_times)
            approx_mean, approx_std, approx_low, approx_high = mean_pm_2std(approximate_times)
            p10_mean, p10_std, p10_low, p10_high = mean_pm_2std(p10s)

            print(f"USER {dataset_size} {m} {ef_construction} {ef_search} Precision@10: {p10_mean:.4f} ± {2*p10_std:.4f} "
                f"(range: {p10_low:.4f} - {p10_high:.4f})")
            
            print(f"USER {dataset_size} {m} {ef_construction} {ef_search} Exact Time: {exact_mean:.4f} ± {2*exact_std:.4f} "
                f"(range: {exact_low:.4f} - {exact_high:.4f})")

            print(f"USER {dataset_size} {m} {ef_construction} {ef_search} Approx Time: {approx_mean:.4f} ± {2*approx_std:.4f} "
                f"(range: {approx_low:.4f} - {approx_high:.4f})")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Precision@10 for dataset size 50000, vector dim 384, m 8, ef_construction 50, ef_search 32


Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

PRODUCT 500000 32 200 128 Precision@10: 0.9900 ± 0.0632 (range: 0.9268 - 1.0532)
PRODUCT 500000 32 200 128 Exact Time: 0.0435 ± 0.0128 (range: 0.0307 - 0.0563)
PRODUCT 500000 32 200 128 Approx Time: 0.0010 ± 0.0008 (range: 0.0002 - 0.0018)


Batches:   0%|          | 0/13028 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

USER 500000 32 200 128 Precision@10: 0.9200 ± 0.2066 (range: 0.7134 - 1.1266)
USER 500000 32 200 128 Exact Time: 0.0411 ± 0.0039 (range: 0.0373 - 0.0450)
USER 500000 32 200 128 Approx Time: 0.0015 ± 0.0018 (range: -0.0004 - 0.0033)
